In [0]:
%sql
-- =============================================================================
-- Notebook: 00_Watermark_Table_Setup
-- Run ONCE to set up the watermark table with Delta optimizations.
-- Idempotent — safe to re-run.
-- =============================================================================

-- Create schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS ingestion_metadata
    COMMENT 'Pipeline watermarks and ingestion metadata';

-- Create the watermark table with Delta best-practice properties
CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
    source_system              STRING    COMMENT 'bigquery | salesforce',
    object_name                STRING    COMMENT 'events | account | contact | etc.',
    last_processed_timestamp   TIMESTAMP COMMENT 'Watermark minus 1 minute for overlap safety'
)
USING DELTA
COMMENT 'Pipeline watermark table — tracks last successfully processed timestamp per source+object.'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',   -- auto bin-pack small files on write
    'delta.autoOptimize.autoCompact'   = 'true',   -- background compaction
    'delta.enableChangeDataFeed'       = 'false',  -- not needed for watermarks
    'delta.dataSkippingNumIndexedCols' = '2'       -- index both PK columns
);

-- Add a composite primary-key constraint (informational, not enforced by Delta)
-- This documents intent and is used by DLT if you migrate later.
-- ALTER TABLE ingestion_metadata.watermarks
--     ADD CONSTRAINT pk_watermarks PRIMARY KEY (source_system, object_name) NOT ENFORCED;

-- Verify
DESCRIBE TABLE EXTENDED ingestion_metadata.watermarks;
